# PISA STUDENT CSV FILTER

In [ ]:
# ------------------------------------------------------------------
# 0.  Imports
# ------------------------------------------------------------------
from pathlib import Path             # nicer than os.path for paths
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1.  File paths and ingestion
# ------------------------------------------------------------------
ROOT          = Path.cwd()
DATAFRAME_PATH = ROOT / "data/pisa_2022/p22_stu_filtered.csv"
CODEBOOK_PATH  = ROOT / "data/pisa_2022/filtered_codebook.csv"

data_df     = pd.read_csv(DATAFRAME_PATH)
codebook_df = pd.read_csv(CODEBOOK_PATH)

print(f"Loaded {len(data_df):,} rows and {data_df.shape[1]} columns")

# Keep only codebook rows that correspond to columns actually present
codebook_df = codebook_df[codebook_df["NAME"].isin(data_df.columns)]

# ------------------------------------------------------------------
# NEW: trim the data to the surviving code‑book variables
# ------------------------------------------------------------------
ID_COLS   = []                      # e.g. ["CNT", "CNTSCHID"] if you still need them
keep_cols = ID_COLS + codebook_df["NAME"].tolist()

data_df   = data_df.loc[:, keep_cols]

# ------------------------------------------------------------------
# 2.  Column‑level filtering (USE == 0 ➜ drop)
# ------------------------------------------------------------------
drop_cols    = codebook_df.loc[codebook_df["USE"] == 0, "NAME"]
data_df      = data_df.drop(columns=drop_cols, errors="ignore")          # returns a *new* DF
codebook_df  = codebook_df[codebook_df["USE"] != 0]                      # keep in sync

# ------------------------------------------------------------------
# 3.  Value cleansing, mode imputation, numerical adjustment
# ------------------------------------------------------------------
for _, row in codebook_df.iterrows():
    col        = row["NAME"]
    min_val    = row["MIN"]
    max_val    = row["MAX"]
    adjust_val = 0 if pd.isna(row.get("ADJUST", 0)) else row["ADJUST"]

    if col not in data_df.columns:                # may have been dropped earlier
        continue

    # (a) coerce to numeric (invalid → NaN)
    data_df[col] = pd.to_numeric(data_df[col], errors="coerce")

    # (b) trim to [min, max]
    outside = (data_df[col] < min_val) | (data_df[col] > max_val)
    data_df.loc[outside, col] = np.nan

    # (c) mode imputation, if the column is not completely NaN
    mode_series = data_df[col].mode(dropna=True)
    if not mode_series.empty:
        mode_val = mode_series.iloc[0]
        data_df[col] = data_df[col].fillna(mode_val)    # ← *no* chained assignment

    # (d) apply ADJUST
    #if adjust_val != 0:
    #    data_df[col] = data_df[col] + adjust_val

# ------------------------------------------------------------------
# 4.  One‑hot / dummy variables for categorical columns
# ------------------------------------------------------------------
categorical_rows = codebook_df[codebook_df["CATEGORICAL"] == 1]
categorical_cols = categorical_rows["NAME"].tolist()

# Restrict to the categorical columns still present
categorical_cols = [c for c in categorical_cols if c in data_df.columns]

# Build the dummy matrix in one go; keeps the DF un‑fragmented
if categorical_cols:
    # Tell pandas the *complete* category range so missing values still get a 0 column
    for _, row in categorical_rows.iterrows():
        col = row["NAME"]
        if col in categorical_cols:                        # guard in case it vanished
            full_range = range(int(row["MIN"]), int(row["MAX"]) + 1)
            data_df[col] = pd.Categorical(data_df[col], categories=full_range)

    dummies = pd.get_dummies(
        data_df[categorical_cols],
        prefix=categorical_cols,
        dtype="int8"                 # saves RAM, still fine for 0/1
    )

    # Drop originals and concat once – avoids fragmentation
    data_df = pd.concat(
        [data_df.drop(columns=categorical_cols), dummies],
        axis=1
    )

print(f"Transformed {len(data_df):,} rows and {data_df.shape[1]} columns")

# ------------------------------------------------------------------
# 5.  Persist the cleaned frame
# ------------------------------------------------------------------
RESULT_PATH = ROOT / "data/pisa_2022/pisa_clean.csv"
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

data_df.to_csv(RESULT_PATH, index=False)
#print(f"Saved cleaned data to {RESULT_PATH}")


Loaded 6,072 rows and 508 columns from c:\Users\jonat\OneDrive\Documents\6.University\()Degree_Project\StudentPerformance_ML\data\pisa_2022\p22_stu_filtered.csv
Transformed 6,072 rows and 521 columns
Saved cleaned data to c:\Users\jonat\OneDrive\Documents\6.University\()Degree_Project\StudentPerformance_ML\data\pisa_2022\pisa_clean.csv
